# Основной бенчмарк

In [ ]:
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
sys.path.append('..')

In [ ]:
from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

def compare_structures(s1, s2, to_print=False):
    # s1 = original_structure
    # s2 = backend.SLICES2structure(slices_NdSiRu)[0]

    ltol=0.3
    stol=0.5
    angle_tol=5

    
    # 2. Compare Lattice parameters manually (optional)
    tol = ltol  # 1% tolerance
    for a, b in zip(s1.lattice.abc, s2.lattice.abc):
        if abs(a-b)/max(a,b) > tol:
            if to_print: print(f"Lattice lengths differ by more than {int(ltol * 100)}%:", s1.lattice.abc, s2.lattice.abc)
            return False
    for α, β in zip(s1.lattice.angles, s2.lattice.angles):
        if abs(α-β) > angle_tol:  # 1° tolerance
            if to_print: print("Lattice angles differ by more than {angle_tol}°:", s1.lattice.angles, s2.lattice.angles)
            return False

    # 3. Compare Space Groups
    # spg1 = SpacegroupAnalyzer(s1).get_space_group_symbol()
    # spg2 = SpacegroupAnalyzer(s2).get_space_group_symbol()
    # if spg1 != spg2:
    #     print(f"Different space groups: {spg1} vs {spg2}")
    # else:
    #     print(f"Same space group: {spg1}")

    # 4. Fuzzy structure matching (accounts for cell reduction, slight distortions,
    #    ordering of sites, and atomic‐type permutations)

    # print(s1.composition)
    # print(s2.composition)

    # print(s2.lattice)
    # print(len(s2.sites))
    # print('---------------------')
    # print(s1.lattice)
    # print(len(s1.sites))

    # if any(not site.is_ordered for site in s2.sites):
    #     print("Partial occupancy detected.")

    # import numpy as np
    # if np.any(np.isnan([c for site in s2.sites for c in site.frac_coords])):
    #     print("NaN in coordinates!")

    matcher = StructureMatcher(
        ltol=ltol,  # lattice length tolerance (5%)
        stol=stol,   # site position tolerance (Å)
        angle_tol=angle_tol,  # angle tolerance (°)
        primitive_cell=True,
        scale=False
    )

    # print(matcher.fit(s2, s2))
    # print('--------------')
    # print(matcher.fit(s1, s1))

    # return False

    are_fit = matcher.fit(s1.get_primitive_structure(), s2.get_primitive_structure())

    if to_print:
        if are_fit:
            print("Structures match (within tolerances) 🎉")
        else:
            print("Structures do not match.")
    return are_fit

In [ ]:
from src.slices_pipeline.encoder import *
from src.slices_pipeline.loader import *
from slices.core import SLICES
from pymatgen.core.structure import Structure
from pymatgen.io.cif import CifParser
from io import StringIO
from tqdm import tqdm
import time
import os

def run_split(split, start_from=0, append_previous_result=True):
    df = read_split_df(split)
    correct = 0
    overall = 0
    error = 0
    file_to_save = "results.txt"
    if (append_previous_result and os.path.exists(file_to_save)) or start_from > 0:
        with open(file_to_save, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if len(lines) > 0:
                if not lines[-1]:
                    lines = lines[:-1]
                num_lines = len(lines)
                correct = sum(1 for l in lines if l[0] == ' ')
                overall = num_lines
                error = sum(1 for l in lines if l[0] == 'E')
                start_from = num_lines
    data = []
    backend = SLICES(relax_model="chgnet", steps=100)
    # backend = SLICES()
    with tqdm(total=df.shape[0], initial=start_from, smoothing=0.9) as pbar:
        for i, cif in enumerate(df.cif.iloc[start_from:], start=start_from):
            start_time = time.time()
            try:
                # original_structure = Structure.from_str(cif, fmt="cif", primitive=True)
                original_structure = CifParser(StringIO(cif)).parse_structures()[0]
                # print(i, df.iloc[i].material_id)
                slices_NdSiRu = backend.structure2SLICES(original_structure)
                reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices_NdSiRu)
                is_okay = compare_structures(original_structure, reconstructed_structure)
                if is_okay: correct += 1
                end_time = time.time()
                data.append(f"{' ' if is_okay else 'x'} {i}: {df.iloc[i]['material_id']} --- {end_time - start_time}")
            except Exception:
                error += 1
                end_time = time.time()
                data.append(f"E {i}: {df.iloc[i]['material_id']} --- {end_time - start_time}")
            finally:
                overall += 1
                pbar.set_postfix(correct=f"{correct}/{overall}", accuracy=f"{correct/overall:.2%}", error=f"{error}")
                pbar.update(1)
                if i % 10 == 9:
                    with open(file_to_save, "a", encoding="utf-8") as f:
                        f.writelines(line + "\n" for line in data)
                    data = []
    with open(file_to_save, "a", encoding="utf-8") as f:
        f.writelines(line + "\n" for line in data)

In [ ]:
run_split("test")

# Анализ

In [ ]:
from src.slices_pipeline.encoder import *
from src.slices_pipeline.loader import *
from slices.core import SLICES

def compare_from_file(cif_file):
    backend = SLICES(relax_model="chgnet")
    original_structure = CifParser(cif_file).get_structures(primitive=False)[0]
    slices_NdSiRu = backend.structure2SLICES(original_structure)
    reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices_NdSiRu)
    is_okay = compare_structures(original_structure, reconstructed_structure, True)
    return is_okay

In [ ]:
df = read_split_df("test")
df.head()

In [ ]:
# compare_from_file("UCo4Sn_mp-13018_primitive.cif")
# compare_from_file("U2FeS5_mp-21037_computed.cif")

In [ ]:
cc = df[df.material_id == "mp-867112"].iloc[0].cif
cc

In [ ]:
compare_from_file(StringIO(cc))

## Dataset consistency

In [ ]:
len(read_split_df("train").material_id.unique()) + len(read_split_df("test").material_id.unique()) + len(read_split_df("val").material_id.unique())